In [1]:
import pandas as pd
import numpy as np

In [2]:
sales = pd.read_csv(r"C:\Users\fa440\OneDrive\Downloads\Datasets csv files\Sales Dataset.csv")

In [3]:
products = pd.read_csv(r"C:\Users\fa440\OneDrive\Downloads\Datasets csv files\Product Dataset.csv")

In [5]:
customers = pd.read_csv(r"C:\Users\fa440\OneDrive\Downloads\Datasets csv files\Customers Dataset.csv")

In [8]:
print("Sales:", sales.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)

Sales: (1194, 12)
Products: (452, 9)
Customers: (100, 12)


In [9]:
# Data Cleaning
sales.columns = sales.columns.str.strip().str.replace(" ", "_")
products.columns = products.columns.str.strip().str.replace(" ", "_")
customers.columns = customers.columns.str.strip().str.replace(" ", "_")

In [10]:
sales.head()

,Order_ID,Amount,Profit,Quantity,Category,Sub-Category,PaymentMode,Order_Date,CustomerName,State,City,Year-Month
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,Miami,2023-06
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2024-12
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,Buffalo,2021-07
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,Miami,2023-06
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2024-12


In [11]:
products.head()

,ProductID,ProductName,Price,CategoryID,Class,ModifyDate,Resistant,IsAllergic,VitalityDays
0,1,Flour - Whole Wheat,74.2988,3,Medium,21:49.2,Durable,Unknown,0
1,2,Cookie Chocolate Chip,91.2329,3,Medium,39:11.0,Unknown,Unknown,0
2,3,Onions,9.1379,9,Medium,11:51.6,Weak,FALSE,111
3,4,"Sauce - Gravy, Au Jus, Mix",54.3055,9,Medium,46:28.9,Durable,Unknown,0
4,5,Artichokes - Jerusalem,65.4771,2,Low,13:35.4,Durable,TRUE,27


In [12]:
customers.head()

,Index,Customer_Id,First_Name,Last_Name,Company,City,Country,Phone_1,Phone_2,Email,Subscription_Date,Website
0,1,DD37Cf93aecA6Dc,Sheryl,Baxter,Rasmussen Group,East Leonard,Chile,229.077.5154,397.884.0519x718,zunigavanessa@smith.info,2020-08-24,http://www.stephenson.com/
1,2,1Ef7b82A4CAAD10,Preston,Lozano,Vega-Gentry,East Jimmychester,Djibouti,5153435776,686-620-1820x944,vmata@colon.com,2021-04-23,http://www.hobbs.com/
2,3,6F94879bDAfE5a6,Roy,Berry,Murillo-Perry,Isabelborough,Antigua and Barbuda,+1-539-402-0259,(496)978-3969x58947,beckycarr@hogan.com,2020-03-25,http://www.lawrence.com/
3,4,5Cef8BFA16c5e3c,Linda,Olsen,"Dominguez, Mcmillan and Donovan",Bensonview,Dominican Republic,001-808-617-6467x12895,+1-813-324-8756,stanleyblackwell@benson.org,2020-06-02,http://www.good-lyons.com/
4,5,053d585Ab6b3159,Joanna,Bender,"Martin, Lang and Andrade",West Priscilla,Slovakia (Slovak Republic),001-234-203-0635x76146,001-199-446-3860x3486,colinalvarado@miles.net,2021-04-17,https://goodwin-ingram.com/


In [14]:
sales.drop_duplicates(inplace=True)
products.drop_duplicates(inplace=True)
customers.drop_duplicates(inplace=True)

In [15]:
                                   # Create customer mapping using names

customers["CustomerName"] = customers["First_Name"] + " " + customers["Last_Name"]

sales["Customer_ID"] = np.arange(len(sales)) % len(customers) + 1

customers["Customer_ID"] = range(1, len(customers) + 1)
sales["ProductID"] = np.arange(len(sales)) % len(products) + 1


In [16]:
# Merge Datasets

merged_df = sales.merge(
    customers[["Customer_ID","CustomerName","Country","City","Subscription_Date"]],
    on="Customer_ID",
    how="left"
)

merged_df = merged_df.merge(
    products[["ProductID","ProductName","Price","CategoryID"]],
    on="ProductID",
    how="left"
)

merged_df.head()


,Order_ID,Amount,Profit,Quantity,Category,Sub-Category,PaymentMode,Order_Date,CustomerName_x,State,...,Year-Month,Customer_ID,ProductID,CustomerName_y,Country,City_y,Subscription_Date,ProductName,Price,CategoryID
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,...,2023-06,1,1,Sheryl Baxter,Chile,East Leonard,2020-08-24,Flour - Whole Wheat,74.2988,3
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,...,2024-12,2,2,Preston Lozano,Djibouti,East Jimmychester,2021-04-23,Cookie Chocolate Chip,91.2329,3
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,...,2021-07,3,3,Roy Berry,Antigua and Barbuda,Isabelborough,2020-03-25,Onions,9.1379,9
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,...,2023-06,4,4,Linda Olsen,Dominican Republic,Bensonview,2020-06-02,"Sauce - Gravy, Au Jus, Mix",54.3055,9
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,...,2024-12,5,5,Joanna Bender,Slovakia (Slovak Republic),West Priscilla,2021-04-17,Artichokes - Jerusalem,65.4771,2


In [17]:
# Feature Engineering

merged_df["Total_Revenue"] = merged_df["Amount"] * merged_df["Quantity"]

merged_df["Profit_Margin"] = (
    merged_df["Profit"] / merged_df["Amount"]
) * 100

customer_revenue = merged_df.groupby("Customer_ID")["Total_Revenue"].transform("sum")
merged_df["Customer_Lifetime_Value"] = customer_revenue

merged_df.head()


,Order_ID,Amount,Profit,Quantity,Category,Sub-Category,PaymentMode,Order_Date,CustomerName_x,State,...,CustomerName_y,Country,City_y,Subscription_Date,ProductName,Price,CategoryID,Total_Revenue,Profit_Margin,Customer_Lifetime_Value
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,...,Sheryl Baxter,Chile,East Leonard,2020-08-24,Flour - Whole Wheat,74.2988,3,48630,13.109192,807306
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,...,Preston Lozano,Djibouti,East Jimmychester,2021-04-23,Cookie Chocolate Chip,91.2329,3,48630,13.109192,817932
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,...,Roy Berry,Antigua and Barbuda,Isabelborough,2020-03-25,Onions,9.1379,9,48630,13.109192,478466
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,...,Linda Olsen,Dominican Republic,Bensonview,2020-06-02,"Sauce - Gravy, Au Jus, Mix",54.3055,9,69650,26.733668,593543
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,...,Joanna Bender,Slovakia (Slovak Republic),West Priscilla,2021-04-17,Artichokes - Jerusalem,65.4771,2,69650,26.733668,788411


In [18]:
# Summary Statistics

print(merged_df[["Total_Revenue",
                 "Profit_Margin",
                 "Customer_Lifetime_Value"]].describe())


       Total_Revenue  Profit_Margin  Customer_Lifetime_Value
count    1194.000000    1194.000000             1.194000e+03
mean    55994.603853      26.057092             6.693683e+05
std     46365.581805      14.051796             1.628057e+05
min       697.000000       0.779018             2.657620e+05
25%     17162.000000      14.066493             5.704870e+05
50%     42732.000000      25.574249             6.610690e+05
75%     85201.500000      38.192527             7.741300e+05
max    199840.000000      50.000000             1.110605e+06


In [20]:
# Save Final Dataset

merged_df.to_csv("Final_Wrangled_Dataset.csv", index=False)

print("Final dataset saved successfully.")


Final dataset saved successfully.


In [21]:
import os
os.getcwd()

'C:\\Users\\fa440\\PyCharmMiscProject'

In [22]:
import os
os.listdir()

['.idea',
 '.venv',
 'Cs-506.ipynb',
 'Customer Segmentation Using Unsupervised Learning.ipynb',
 'Energy Consumption Time Series Forecasting.ipynb',
 'Final_Wrangled_Dataset.csv',
 'notebook.ipynb',
 'Practice test 1.py',
 'script.py',
 'Term Deposit Subscription Prediction (Bank Marketing).ipynb']